Downloading dependencies

In [48]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

from sklearn.model_selection import train_test_split

In [49]:
df_obesity = pd.read_csv("obesity.csv")

Analyzing df

In [50]:
df_obesity.head()

,id,Gender,Age,Height,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS,NObeyesdad
0,0,Male,24.443011,1.699998,81.669950,yes,yes,2.000000,2.983297,Sometimes,no,2.763573,no,0.000000,0.976473,Sometimes,Public_Transportation,Overweight_Level_II
1,1,Female,18.000000,1.560000,57.000000,yes,yes,2.000000,3.000000,Frequently,no,2.000000,no,1.000000,1.000000,no,Automobile,Normal_Weight
2,2,Female,18.000000,1.711460,50.165754,yes,yes,1.880534,1.411685,Sometimes,no,1.910378,no,0.866045,1.673584,no,Public_Transportation,Insufficient_Weight
3,3,Female,20.952737,1.710730,131.274851,yes,yes,3.000000,3.000000,Sometimes,no,1.674061,no,1.467863,0.780199,Sometimes,Public_Transportation,Obesity_Type_III
4,4,Male,31.641081,1.914186,93.798055,yes,yes,2.679664,1.971472,Sometimes,no,1.979848,no,1.967973,0.931721,Sometimes,Public_Transportation,Overweight_Level_II


In [51]:
df_obesity.describe()

,id,Age,Height,Weight,FCVC,NCP,CH2O,FAF,TUE
count,20758.00000,20758.000000,20758.000000,20758.000000,20758.000000,20758.000000,20758.000000,20758.000000,20758.000000
mean,10378.50000,23.841804,1.700245,87.887768,2.445908,2.761332,2.029418,0.981747,0.616756
std,5992.46278,5.688072,0.087312,26.379443,0.533218,0.705375,0.608467,0.838302,0.602113
min,0.00000,14.000000,1.450000,39.000000,1.000000,1.000000,1.000000,0.000000,0.000000
25%,5189.25000,20.000000,1.631856,66.000000,2.000000,3.000000,1.792022,0.008013,0.000000
50%,10378.50000,22.815416,1.700000,84.064875,2.393837,3.000000,2.000000,1.000000,0.573887
75%,15567.75000,26.000000,1.762887,111.600553,3.000000,3.000000,2.549617,1.587406,1.000000
max,20757.00000,61.000000,1.975663,165.057269,3.000000,4.000000,3.000000,3.000000,2.000000


Checking for nullity

In [52]:
df_obesity.isnull().sum()

id                                0
Gender                            0
Age                               0
Height                            0
Weight                            0
family_history_with_overweight    0
FAVC                              0
FCVC                              0
NCP                               0
CAEC                              0
SMOKE                             0
CH2O                              0
SCC                               0
FAF                               0
TUE                               0
CALC                              0
MTRANS                            0
NObeyesdad                        0
dtype: int64

Defining categorical features with only 2 uniq variables, and checking what values they contain

In [53]:
categorical_features = df_obesity.select_dtypes(include=["str"]).columns.tolist()
df_obesity[categorical_features].nunique()

categorical_features_values = []
indexes = []

for i in categorical_features:
    if df_obesity[i].nunique() == 2:
        categorical_features_values.append(df_obesity[i].value_counts())
        indexes.append(i)
categorical_features_values = pd.DataFrame(categorical_features_values)
categorical_features_values.set_index([indexes], inplace=True)
categorical_features_values

,Female,Male,yes,no
Gender,10422.0,10336.0,NaN,NaN
family_history_with_overweight,NaN,NaN,17014.0,3744.0
FAVC,NaN,NaN,18982.0,1776.0
SMOKE,NaN,NaN,245.0,20513.0
SCC,NaN,NaN,687.0,20071.0


Normalizing yes/no and male/female features to 0/1

In [54]:
df_obesity["Gender"] = df_obesity["Gender"].apply(lambda x: 0 if x == "Male" else 1)

for i in ["family_history_with_overweight","FAVC","SMOKE","SCC"]:
    df_obesity[i] = df_obesity[i].apply(lambda x: 0 if x == "no" else 1)

Dividing into test and training sets

In [55]:
target_col = "NObeyesdad"
y = df_obesity[target_col]
X = df_obesity.drop(columns=[target_col])

One hot encoding for remained values with more than 2 variables

In [56]:
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["str"]).columns.tolist()

In [57]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

Dividing for test and training using a 80/20 fraction

In [58]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

Adding different models and their params

In [59]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

In [60]:
models = {
    "LogisticRegression": LogisticRegression(max_iter=2000),
    "RandomForest": RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        random_state=42,
        n_jobs=-1
    ),
    "GradientBoosting": GradientBoostingClassifier(random_state=42),
}

results = []

Training and predicting using models, adding preprocessor from one hot encoding above

In [61]:
for name, model in models.items():
    pipe = Pipeline(steps=[
        ("preprocess", preprocessor),
        ("model", model)
    ])

    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    results.append({"model": name, "accuracy": acc})

C:\Users\Alily\PycharmProjects\ObesityLevel_classification_model\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Accuracy results for all three models

In [62]:
pd.DataFrame(results)

,model,accuracy
0,LogisticRegression,0.641378
1,RandomForest,0.901975
2,GradientBoosting,0.906551
